In [ ]:
import pandas as pd

excel_path = "ruta"


df_pandas = pd.read_excel(excel_path, sheet_name="Inventory Management")

df_spark = spark.createDataFrame(df_pandas)
df_spark.show()


StatementMeta(, 355749d5-f945-400b-aebb-10e4c2a31fda, 3, Finished, Available, Finished)

+-----------+---------------+---------------+---------------+-------------------+----------------+-------------+------------+-----------------------+------------+-----------------+----------------------+------------+-------------------+--------------+-------+--------+
|ID_producto|Nombre_Producto|      Categoria|Precio_Unitario|Cantidad_Inventario|Nivel_Inventario|Punto_Reorden|Días_Entrega|Fecha_Ultima_Reposición|ID_Proveedor|Ubicacion_Almacen|Cantidad_Minima_Pedido|      Estado|      Fecha_Entrada|          Pais|Latitud|Longitud|
+-----------+---------------+---------------+---------------+-------------------+----------------+-------------+------------+-----------------------+------------+-----------------+----------------------+------------+-------------------+--------------+-------+--------+
|  SKU001158|   Product_1158|           Toys|         294.54|                879|            High|           30|          20|    2025-04-10 00:00:00|      SUP024|AISLE-16-SHELF-14|             

**CAPA SILVER**

In [ ]:

from pyspark.sql.functions import count
df_duplicates = df_spark.groupBy("ID_producto").agg(count("*").alias("conteo"))

df_duplicates = df_duplicates.filter("conteo > 1")

df_duplicates.show()


StatementMeta(, 355749d5-f945-400b-aebb-10e4c2a31fda, 6, Finished, Available, Finished)

+-----------+------+
|ID_producto|conteo|
+-----------+------+
+-----------+------+



**Encontrar vacios o nulos**

In [ ]:
from pyspark.sql.functions import col, trim, sum as spark_sum

# df_spark.select([
#     sum((col(c).isNull() | (trim(col(c)) == "")).cast("int")).alias(c + "_vacios_o_nulos")
#     for c in df_spark.columns
# ]).show()

df_spark.select([
    spark_sum((col(c).isNull() | (trim(col(c)) == "")).cast("int")).alias(c + "_vacios_o_nulos")
    for c in df_spark.columns
]).show()

StatementMeta(, 355749d5-f945-400b-aebb-10e4c2a31fda, 12, Finished, Available, Finished)

+--------------------------+------------------------------+------------------------+------------------------------+----------------------------------+-------------------------------+----------------------------+---------------------------+--------------------------------------+---------------------------+--------------------------------+-------------------------------------+---------------------+----------------------------+-------------------+----------------------+-----------------------+
|ID_producto_vacios_o_nulos|Nombre_Producto_vacios_o_nulos|Categoria_vacios_o_nulos|Precio_Unitario_vacios_o_nulos|Cantidad_Inventario_vacios_o_nulos|Nivel_Inventario_vacios_o_nulos|Punto_Reorden_vacios_o_nulos|Días_Entrega_vacios_o_nulos|Fecha_Ultima_Reposición_vacios_o_nulos|ID_Proveedor_vacios_o_nulos|Ubicacion_Almacen_vacios_o_nulos|Cantidad_Minima_Pedido_vacios_o_nulos|Estado_vacios_o_nulos|Fecha_Entrada_vacios_o_nulos|Pais_vacios_o_nulos|Latitud_vacios_o_nulos|Longitud_vacios_o_nulos|
+-------

IDENTIFICAR FECHAS VACIAS

In [ ]:
df_spark.select("Fecha_Ultima_Reposición", "Fecha_Entrada").printSchema()

StatementMeta(, 355749d5-f945-400b-aebb-10e4c2a31fda, 13, Finished, Available, Finished)

root
 |-- Fecha_Ultima_Reposición: timestamp (nullable = true)
 |-- Fecha_Entrada: timestamp (nullable = true)



In [ ]:
from pyspark.sql.functions import col, sum

df_spark.select([
    sum(col(c).isNull().cast("int")).alias(c + "_nulas")
    for c in ["Fecha_Ultima_Reposición", "Fecha_Entrada"]
]).show()


StatementMeta(, 355749d5-f945-400b-aebb-10e4c2a31fda, 14, Finished, Available, Finished)

+-----------------------------+-------------------+
|Fecha_Ultima_Reposición_nulas|Fecha_Entrada_nulas|
+-----------------------------+-------------------+
|                            0|                  0|
+-----------------------------+-------------------+



**VALIDAR NOMBRES DE LAS COLUMNAS**

In [ ]:
df_spark.columns

StatementMeta(, 355749d5-f945-400b-aebb-10e4c2a31fda, 15, Finished, Available, Finished)

['ID_producto',
 'Nombre_Producto',
 'Categoria',
 'Precio_Unitario',
 'Cantidad_Inventario',
 'Nivel_Inventario',
 'Punto_Reorden',
 'Días_Entrega',
 'Fecha_Ultima_Reposición',
 'ID_Proveedor',
 'Ubicacion_Almacen',
 'Cantidad_Minima_Pedido',
 'Estado',
 'Fecha_Entrada',
 'Pais',
 'Latitud',
 'Longitud']

CREANDO TABLA EN EL DWH SILVER

In [ ]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants  

df_spark.write \
    .mode("overwrite") \
    .synapsesql("DWH_Silver_Gold.Silver.tabla_completa_inventario")


StatementMeta(, 355749d5-f945-400b-aebb-10e4c2a31fda, 16, Finished, Available, Finished)

In [ ]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

df_dwh = spark.read.synapsesql("DWH_Silver_Gold.Silver.tabla_completa_inventario")

df_dwh.show()


StatementMeta(, 355749d5-f945-400b-aebb-10e4c2a31fda, 17, Finished, Available, Finished)

+-----------+------------+-----------------------+--------------+----------------------+------------+--------+-----------+-------+-------------------+---------------+-------------------+-------------+----------------+---------------+-----------------+------------+
|  Categoria|      Estado|Fecha_Ultima_Reposición|          Pais|Cantidad_Minima_Pedido|ID_Proveedor|Longitud|ID_producto|Latitud|      Fecha_Entrada|Precio_Unitario|Cantidad_Inventario|Punto_Reorden|Nivel_Inventario|Nombre_Producto|Ubicacion_Almacen|Días_Entrega|
+-----------+------------+-----------------------+--------------+----------------------+------------+--------+-----------+-------+-------------------+---------------+-------------------+-------------+----------------+---------------+-----------------+------------+
|     Sports|Out of Stock|    2025-09-07 00:00:00|       Germany|                    28|      SUP013| 10.9666|  SKU003431| 51.182|2025-03-18 00:00:00|         250.52|                164|           54|     